# LAB 2 — Azure Services & Shared Lakehouse Setup (Week 2)

I used widgets for parameters that might differ between dev and production environments. I also used checkpoints to ensure idempotence.

## Setup

In [0]:
dbutils.widgets.text("catalog", "", "Catalog: ")
dbutils.widgets.text("schema", "", "Schema: ")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = "raw_data"

raw_data_path = f"/Volumes/{catalog}/{schema}/{volume}/"
checkpoint_path = f"/Volumes/{catalog}/{schema}/{volume}/_checkpoints/"

table_name = f"{catalog}.{schema}.co2_bronze"

## Reading raw source file 

In [0]:
from pyspark.sql.functions import input_file_name, current_timestamp, current_date, col

raw_stream_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema_inference") 
    .load(raw_data_path)
)

## Adding metadata

In [0]:

bronze_stream_df = (raw_stream_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file")
    )
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

## Saving as Delta

In [0]:

sq = (bronze_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(table_name)
)

sq.awaitTermination()